In [0]:
silver_df = spark.table("job_marketplace.silver_layer.clean_jobs")
display(silver_df)

## Nucha/Nucha_SkillNER_BERT

In [0]:
# # Step 2: Import libraries
# from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
# from pyspark.sql.functions import col, explode, udf
# from pyspark.sql.types import ArrayType, StringType

# # Step 3: Load your model once
# model_name = "Nucha/Nucha_SkillNER_BERT"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForTokenClassification.from_pretrained(model_name)

# # Create NER pipeline
# ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

In [0]:
jobs_pdf = spark.read.table("job_marketplace.silver_layer.clean_jobs")\
            .filter(col('search_role')=='Data Engineer').limit(100).toPandas()

from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
model_name = "Nucha/Nucha_SkillNER_BERT"
ner = pipeline("ner",
               model=AutoModelForTokenClassification.from_pretrained(model_name),
               tokenizer=AutoTokenizer.from_pretrained(model_name),
               aggregation_strategy="simple")

def extract_skills(text):
    if not text: return []
    results = ner(text)
    return list({r["word"] for r in results})

jobs_pdf["skills"] = jobs_pdf["description"].apply(extract_skills)
# spark.createDataFrame(jobs_pdf[["job_id","skills"]]).write.mode("overwrite").saveAsTable("gold_layer.temp_skills")
display(jobs_pdf)